# Demo NLP: Sequence Labeling cho câu lệnh âm nhạc

Notebook này minh họa bài toán **sequence labeling**: mỗi token trong câu sẽ được gán một nhãn riêng. Thay vì phân loại cả câu là tích cực/tiêu cực, mô hình phải dự đoán nhãn cho từng từ.

Ví dụ:

| Token | mở | bài | Creep | của | Radiohead |
|---|---|---|---|---|---|
| Tag | O | O | B-SONG | O | B-ARTIST |

Trong demo này, ta xây dựng một bộ dữ liệu nhỏ về **câu lệnh nghe nhạc**, sau đó huấn luyện hai mô hình:

1. **Hidden Markov Model (HMM)**
2. **Conditional Random Fields (CRF)**

Mục tiêu là nhận diện các thực thể như bài hát, nghệ sĩ, thể loại, album và hành động trong câu.

In [ ]:
# Cài thư viện cần thiết
# Nếu chạy trên Google Colab, cell này sẽ tự cài sklearn-crfsuite.
!pip install -q sklearn-crfsuite nltk

## 1. Tạo dữ liệu mẫu

Mỗi câu được biểu diễn dưới dạng danh sách các cặp `(token, tag)`. Các nhãn sử dụng kiểu BIO:

- `B-SONG`: token bắt đầu tên bài hát
- `I-SONG`: token nằm bên trong tên bài hát
- `B-ARTIST`: token bắt đầu tên nghệ sĩ
- `I-ARTIST`: token nằm bên trong tên nghệ sĩ
- `B-GENRE`: token bắt đầu tên thể loại
- `B-ALBUM`: token bắt đầu tên album
- `I-ALBUM`: token nằm bên trong tên album
- `B-ACTION`: hành động như mở, phát, tìm, thêm
- `O`: token bình thường, không thuộc thực thể cần nhận diện

In [ ]:
from pprint import pprint

# Dataset nhỏ, tự xây dựng cho demo sequence labeling chủ đề âm nhạc
train_sents = [
    [("mở", "B-ACTION"), ("bài", "O"), ("Creep", "B-SONG"), ("của", "O"), ("Radiohead", "B-ARTIST")],
    [("phát", "B-ACTION"), ("Smells", "B-SONG"), ("Like", "I-SONG"), ("Teen", "I-SONG"), ("Spirit", "I-SONG"), ("của", "O"), ("Nirvana", "B-ARTIST")],
    [("nghe", "B-ACTION"), ("Comfortably", "B-SONG"), ("Numb", "I-SONG"), ("của", "O"), ("Pink", "B-ARTIST"), ("Floyd", "I-ARTIST")],
    [("tìm", "B-ACTION"), ("bài", "O"), ("Fake", "B-SONG"), ("Plastic", "I-SONG"), ("Trees", "I-SONG")],
    [("thêm", "B-ACTION"), ("Wonderwall", "B-SONG"), ("vào", "O"), ("playlist", "O")],
    [("mở", "B-ACTION"), ("album", "O"), ("OK", "B-ALBUM"), ("Computer", "I-ALBUM"), ("của", "O"), ("Radiohead", "B-ARTIST")],
    [("phát", "B-ACTION"), ("nhạc", "O"), ("rock", "B-GENRE"), ("buổi", "O"), ("tối", "O")],
    [("nghe", "B-ACTION"), ("nhạc", "O"), ("jazz", "B-GENRE"), ("nhẹ", "O"), ("nhàng", "O")],
    [("tìm", "B-ACTION"), ("playlist", "O"), ("indie", "B-GENRE"), ("cho", "O"), ("cuối", "O"), ("tuần", "O")],
    [("phát", "B-ACTION"), ("Bohemian", "B-SONG"), ("Rhapsody", "I-SONG"), ("của", "O"), ("Queen", "B-ARTIST")],
    [("mở", "B-ACTION"), ("Hotel", "B-SONG"), ("California", "I-SONG"), ("của", "O"), ("Eagles", "B-ARTIST")],
    [("nghe", "B-ACTION"), ("Black", "B-SONG"), ("Hole", "I-SONG"), ("Sun", "I-SONG"), ("của", "O"), ("Soundgarden", "B-ARTIST")],
    [("tìm", "B-ACTION"), ("album", "O"), ("The", "B-ALBUM"), ("Wall", "I-ALBUM")],
    [("mở", "B-ACTION"), ("nhạc", "O"), ("acoustic", "B-GENRE"), ("để", "O"), ("học", "O"), ("bài", "O")],
    [("phát", "B-ACTION"), ("nhạc", "O"), ("metal", "B-GENRE"), ("của", "O"), ("Metallica", "B-ARTIST")],
    [("nghe", "B-ACTION"), ("Nothing", "B-SONG"), ("Else", "I-SONG"), ("Matters", "I-SONG")],
    [("tìm", "B-ACTION"), ("ca", "O"), ("sĩ", "O"), ("Thom", "B-ARTIST"), ("Yorke", "I-ARTIST")],
    [("mở", "B-ACTION"), ("No", "B-SONG"), ("Surprises", "I-SONG"), ("của", "O"), ("Radiohead", "B-ARTIST")],
    [("phát", "B-ACTION"), ("Come", "B-SONG"), ("As", "I-SONG"), ("You", "I-SONG"), ("Are", "I-SONG")],
    [("nghe", "B-ACTION"), ("nhạc", "O"), ("blues", "B-GENRE"), ("khi", "O"), ("trời", "O"), ("mưa", "O")],
    [("thêm", "B-ACTION"), ("Wish", "B-SONG"), ("You", "I-SONG"), ("Were", "I-SONG"), ("Here", "I-SONG"), ("vào", "O"), ("playlist", "O")],
    [("mở", "B-ACTION"), ("album", "O"), ("Nevermind", "B-ALBUM"), ("của", "O"), ("Nirvana", "B-ARTIST")],
    [("tìm", "B-ACTION"), ("bài", "O"), ("High", "B-SONG"), ("and", "I-SONG"), ("Dry", "I-SONG")],
    [("phát", "B-ACTION"), ("nhạc", "O"), ("pop", "B-GENRE"), ("vui", "O"), ("vẻ", "O")],
]

test_sents = [
    [("mở", "B-ACTION"), ("Paranoid", "B-SONG"), ("Android", "I-SONG"), ("của", "O"), ("Radiohead", "B-ARTIST")],
    [("phát", "B-ACTION"), ("Lithium", "B-SONG"), ("của", "O"), ("Nirvana", "B-ARTIST")],
    [("nghe", "B-ACTION"), ("nhạc", "O"), ("rock", "B-GENRE"), ("cổ", "O"), ("điển", "O")],
    [("tìm", "B-ACTION"), ("album", "O"), ("In", "B-ALBUM"), ("Rainbows", "I-ALBUM")],
    [("thêm", "B-ACTION"), ("Karma", "B-SONG"), ("Police", "I-SONG"), ("vào", "O"), ("playlist", "O")],
]

print("Số câu train:", len(train_sents))
print("Số câu test:", len(test_sents))
print("
Ví dụ một câu đã gán nhãn:")
pprint(train_sents[0])

## 2. Huấn luyện mô hình Hidden Markov Model

HMM là mô hình xác suất cho dữ liệu chuỗi. Trong sequence labeling, mô hình cố gắng tìm chuỗi nhãn hợp lý nhất cho chuỗi token đầu vào.

Ý tưởng đơn giản:

- Một nhãn có thể phụ thuộc vào nhãn đứng trước nó.
- Một từ có xác suất xuất hiện khác nhau ở từng loại nhãn.
- Khi dự đoán, HMM chọn chuỗi nhãn có xác suất cao nhất.

In [ ]:
from nltk.tag import HiddenMarkovModelTagger

print("Đang huấn luyện HMM...")
hmm_model = HiddenMarkovModelTagger.train(train_sents)

# Dự đoán trên tập test
y_true_hmm = []
y_pred_hmm = []

for sent in test_sents:
    words = [word for word, tag in sent]
    true_tags = [tag for word, tag in sent]
    pred_pairs = hmm_model.tag(words)
    pred_tags = [tag for word, tag in pred_pairs]

    y_true_hmm.append(true_tags)
    y_pred_hmm.append(pred_tags)

print("Ví dụ dự đoán HMM:")
for word, tag in hmm_model.tag(["mở", "Creep", "của", "Radiohead"]):
    print(f"{word:12s} -> {tag}")

## 3. Xây dựng đặc trưng cho CRF

CRF không chỉ nhìn token hiện tại, mà còn có thể dùng thêm thông tin ngữ cảnh xung quanh token đó.

Một số đặc trưng dùng trong demo:

- Dạng chữ thường của token
- Token có viết hoa hay không
- Token có phải chữ số không
- Tiền tố và hậu tố của token
- Từ đứng trước và từ đứng sau
- Vị trí đầu câu hoặc cuối câu

Nhờ vậy, CRF thường ổn định hơn HMM trên các bài toán sequence labeling truyền thống.

In [ ]:
def word2features(sent, i):
    # Trích xuất đặc trưng cho token thứ i trong một câu đã gán nhãn.
    word = sent[i][0]

    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word[:2]": word[:2],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
        "token_length": len(word),
    }

    if i > 0:
        prev_word = sent[i - 1][0]
        features.update({
            "-1:word.lower()": prev_word.lower(),
            "-1:word.istitle()": prev_word.istitle(),
            "-1:word.isupper()": prev_word.isupper(),
        })
    else:
        features["BOS"] = True  # Beginning of sentence

    if i < len(sent) - 1:
        next_word = sent[i + 1][0]
        features.update({
            "+1:word.lower()": next_word.lower(),
            "+1:word.istitle()": next_word.istitle(),
            "+1:word.isupper()": next_word.isupper(),
        })
    else:
        features["EOS"] = True  # End of sentence

    return features


def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]


def sent2labels(sent):
    return [label for token, label in sent]


def sent2tokens(sent):
    return [token for token, label in sent]

X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]
X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

print("Token trong câu đầu tiên:")
print(sent2tokens(train_sents[0]))
print("
Đặc trưng của token đầu tiên:")
pprint(X_train[0][0])

## 4. Huấn luyện mô hình CRF

CRF học mối quan hệ giữa token, đặc trưng của token và nhãn tương ứng. Trong bài toán này, CRF phù hợp vì nhãn của các token thường có quan hệ với nhau.

Ví dụ: sau `B-SONG`, token tiếp theo có thể là `I-SONG` nếu tên bài hát gồm nhiều từ.

In [ ]:
import sklearn_crfsuite

print("Đang huấn luyện CRF...")
crf_model = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
)

crf_model.fit(X_train, y_train)
y_pred_crf = crf_model.predict(X_test)

print("Huấn luyện xong CRF.")
print("Các nhãn mà mô hình học được:")
print(crf_model.classes_)

## 5. Đánh giá mô hình

Ta dùng `flat_classification_report` để đánh giá kết quả trên từng token.

Các chỉ số chính:

- **Precision**: trong các token được dự đoán là một nhãn, có bao nhiêu token đúng.
- **Recall**: trong các token thật sự thuộc một nhãn, mô hình tìm được bao nhiêu token.
- **F1-score**: trung bình điều hòa giữa precision và recall.

Vì dataset nhỏ nên kết quả chỉ mang tính minh họa, không nên xem là độ chính xác thực tế khi áp dụng ngoài đời.

In [ ]:
from sklearn_crfsuite import metrics

all_labels = sorted(list(set(tag for sent in y_test for tag in sent) | set(tag for sent in y_pred_crf for tag in sent)))
labels_without_o = [label for label in all_labels if label != "O"]

print("--- KẾT QUẢ HMM ---")
print(metrics.flat_classification_report(y_true_hmm, y_pred_hmm, labels=labels_without_o, zero_division=0))

print("
--- KẾT QUẢ CRF ---")
print(metrics.flat_classification_report(y_test, y_pred_crf, labels=labels_without_o, zero_division=0))

## 6. Xem dự đoán chi tiết

Phần này in từng token kèm nhãn thật và nhãn dự đoán để dễ quan sát mô hình sai ở đâu.

In [ ]:
def show_prediction(sent, pred_tags):
    print(f"{'TOKEN':15s} {'TRUE':12s} {'PRED':12s}")
    print("-" * 42)
    for (token, true_tag), pred_tag in zip(sent, pred_tags):
        print(f"{token:15s} {true_tag:12s} {pred_tag:12s}")

for idx in range(len(test_sents)):
    print(f"
Câu test {idx + 1}:")
    show_prediction(test_sents[idx], y_pred_crf[idx])

## 7. Dự đoán câu mới

Hàm dưới đây nhận một câu dạng text, tách token đơn giản bằng khoảng trắng, sau đó dùng CRF để dự đoán nhãn cho từng token.

In [ ]:
def predict_sequence(text, model=crf_model):
    tokens = text.split()
    sent = [(token, "O") for token in tokens]  # tag giả để dùng lại hàm trích đặc trưng
    features = sent2features(sent)
    pred_tags = model.predict_single(features)
    return list(zip(tokens, pred_tags))

examples = [
    "mở Creep của Radiohead",
    "phát nhạc rock buổi sáng",
    "tìm album OK Computer",
    "thêm Smells Like Teen Spirit vào playlist",
]

for text in examples:
    print("
Input:", text)
    for token, tag in predict_sequence(text):
        print(f"{token:12s} -> {tag}")

## 8. Kết luận

Notebook này đã minh họa một pipeline sequence labeling cơ bản:

```text
Dữ liệu câu đã gán nhãn
→ Tách token và nhãn
→ Huấn luyện HMM
→ Trích xuất đặc trưng
→ Huấn luyện CRF
→ Đánh giá theo từng token
→ Dự đoán nhãn cho câu mới
```

Trong demo này, HMM và CRF đều có thể dùng để gán nhãn chuỗi. Tuy nhiên, CRF linh hoạt hơn vì có thể tận dụng nhiều đặc trưng của token và ngữ cảnh xung quanh. Với dữ liệu lớn hơn, bài toán này có thể mở rộng thành các hệ thống nhận diện thực thể âm nhạc như tên bài hát, tên nghệ sĩ, album hoặc thể loại nhạc trong câu lệnh của người dùng.